In [1]:
# ============================================================
# Patch extraction test pipeline
# ------------------------------------------------------------
# 목적:
#   - 전처리 결과를 읽어서 patch 후보 생성
#   - pure_lung 기준으로 폐 안쪽 patch만 선택
#   - 위치 그룹 부여: upper/middle/lower × central/peripheral
#   - left/right 정보는 metadata에만 저장
#   - HU 기반 간단 feature 계산
#   - patch metadata CSV 저장
#
# 전처리 코드는 건드리지 않음
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import SimpleITK as sitk
from tqdm import tqdm
from scipy.ndimage import distance_transform_edt


# ============================================================
# 1. CONFIG
# ============================================================

PATCH_CONFIG = {
    "preprocess_root": r"E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_final_all_nonfast_v1",
    "out_dir_name": "patch_all_v1",

    # patch 크기
    # 32×32 네모 조각으로 자름
    "patch_size": 32,

    # 16칸씩 이동
    # 즉, patch가 반 정도 겹치게 자름
    "patch_stride": 16,

    # slice 자체에 폐가 거의 없으면 제외
    "min_slice_pure_lung_ratio": 0.01,

    # 학습용 정상 기준 후보 patch
    # patch 안에서 pure_lung이 70% 이상이어야 함
    "min_patch_pure_lung_ratio": 0.70,

    # organ exclusion이 너무 많이 겹치면 제외
    "max_patch_organ_ratio": 0.02,

    # 중심부/말초부 기준
    # 폐 경계에서 가장 먼 거리 대비 0.5 이상이면 central
    "central_distance_ratio_threshold": 0.50,

    # 너무 작은 폐 mask slice는 distance map이 불안정하므로 제외
    "min_pure_lung_pixels_per_slice": 100,

    # HU feature용 histogram bin
    "hist_bins": [-1000, -900, -800, -700, -600, -500, -400, -300, -200, -100, 0, 100, 200, 400],
}

args = PATCH_CONFIG

PRE_ROOT = Path(args["preprocess_root"])
PATCH_OUT = PRE_ROOT / args["out_dir_name"]
PATCH_OUT.mkdir(parents=True, exist_ok=True)

with open(PATCH_OUT / "patch_config.json", "w", encoding="utf-8") as f:
    json.dump(PATCH_CONFIG, f, indent=2, ensure_ascii=False)

print("PRE_ROOT:", PRE_ROOT)
print("PATCH_OUT:", PATCH_OUT)

PRE_ROOT: E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_final_all_nonfast_v1
PATCH_OUT: E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_final_all_nonfast_v1\patch_all_v1


In [2]:
# ============================================================
# 2. 기본 함수
# ============================================================

def read_nii_array(path):
    img = sitk.ReadImage(str(path))
    arr = sitk.GetArrayFromImage(img)
    return img, arr


def calc_patch_features(patch_hu, hist_bins):
    """
    patch 안 HU 값으로 간단 feature 계산.
    처음 실험용이라 복잡한 CNN feature는 안 씀.
    """

    x = patch_hu.astype(np.float32).reshape(-1)

    hist, _ = np.histogram(x, bins=hist_bins)
    hist = hist.astype(np.float32)
    hist_sum = hist.sum()

    if hist_sum > 0:
        hist = hist / hist_sum

    feat = {
        "hu_mean": float(np.mean(x)),
        "hu_std": float(np.std(x)),
        "hu_min": float(np.min(x)),
        "hu_max": float(np.max(x)),
        "hu_p05": float(np.percentile(x, 5)),
        "hu_p25": float(np.percentile(x, 25)),
        "hu_p50": float(np.percentile(x, 50)),
        "hu_p75": float(np.percentile(x, 75)),
        "hu_p95": float(np.percentile(x, 95)),
    }

    # histogram feature도 같이 저장
    for i, v in enumerate(hist):
        feat[f"hist_bin_{i:02d}"] = float(v)

    return feat


def get_z_level(z, valid_z_min, valid_z_max):
    """
    z축 위치를 upper / middle / lower로 나눔.
    절대 slice 번호가 아니라, 폐가 있는 범위 안에서 상대 위치로 계산.
    """

    if valid_z_max <= valid_z_min:
        return "middle", 0.5

    z_ratio = (z - valid_z_min) / float(valid_z_max - valid_z_min)
    z_ratio = float(np.clip(z_ratio, 0.0, 1.0))

    if z_ratio < 1.0 / 3.0:
        return "lower", z_ratio
    elif z_ratio < 2.0 / 3.0:
        return "middle", z_ratio
    else:
        return "upper", z_ratio


def get_central_peripheral(dist_patch_values, threshold=0.5):
    """
    patch 안 pure_lung 영역의 distance ratio 평균으로 central/peripheral 판단.
    patch 중심점 하나만 쓰는 것보다 조금 더 안정적임.
    """

    if len(dist_patch_values) == 0:
        return "peripheral", 0.0

    mean_ratio = float(np.mean(dist_patch_values))

    if mean_ratio >= threshold:
        return "central", mean_ratio

    return "peripheral", mean_ratio


def get_left_right_by_patch_center(x0, patch_size, width):
    """
    좌우 정보는 일단 metadata용으로만 저장.
    이미지 기준 왼쪽/오른쪽임.
    의학적 left/right와 완전히 같다고 단정하지 않음.
    """

    cx = x0 + patch_size / 2.0

    if cx < width / 2.0:
        return "image_left"

    return "image_right"

In [3]:
# ============================================================
# 3. patch 추출 실행
# ============================================================

metadata_path = PRE_ROOT / "metadata_slices.csv"

if not metadata_path.exists():
    raise FileNotFoundError(f"metadata_slices.csv 없음: {metadata_path}")

slice_df = pd.read_csv(metadata_path)

required_cols = [
    "patient_id",
    "ct_1mm_lung_range_nii",
    "pure_lung_lung_range_nii",
    "organ_exclusion_lung_range_nii",
]

for c in required_cols:
    if c not in slice_df.columns:
        raise RuntimeError(f"metadata_slices.csv에 필요한 컬럼 없음: {c}")

patient_ids = sorted(slice_df["patient_id"].unique())

print("patient count:", len(patient_ids))
print("patients:", patient_ids)


all_patch_rows = []
patient_summary_rows = []

patch_size = int(args["patch_size"])
patch_stride = int(args["patch_stride"])
patch_area = patch_size * patch_size

for patient_id in tqdm(patient_ids, desc="Patch extraction by patient"):
    pdf = slice_df[slice_df["patient_id"] == patient_id].copy()
    row0 = pdf.iloc[0]

    ct_path = Path(row0["ct_1mm_lung_range_nii"])
    pure_path = Path(row0["pure_lung_lung_range_nii"])
    organ_path = Path(row0["organ_exclusion_lung_range_nii"])

    if not ct_path.exists():
        raise FileNotFoundError(f"CT lung range 없음: {ct_path}")
    if not pure_path.exists():
        raise FileNotFoundError(f"pure lung range 없음: {pure_path}")
    if not organ_path.exists():
        raise FileNotFoundError(f"organ exclusion range 없음: {organ_path}")

    ct_img, ct_arr = read_nii_array(ct_path)
    pure_img, pure_arr = read_nii_array(pure_path)
    organ_img, organ_arr = read_nii_array(organ_path)

    ct_arr = ct_arr.astype(np.float32)
    pure_arr = pure_arr > 0
    organ_arr = organ_arr > 0

    if ct_arr.shape != pure_arr.shape:
        raise RuntimeError(f"{patient_id}: CT와 pure_lung shape 다름")
    if ct_arr.shape != organ_arr.shape:
        raise RuntimeError(f"{patient_id}: CT와 organ_exclusion shape 다름")

    zdim, h, w = ct_arr.shape

    # slice별 pure lung ratio
    pure_area_per_slice = pure_arr.sum(axis=(1, 2))
    pure_ratio_per_slice = pure_area_per_slice / float(h * w)

    valid_z_indices = np.where(
        (pure_ratio_per_slice >= float(args["min_slice_pure_lung_ratio"]))
        & (pure_area_per_slice >= int(args["min_pure_lung_pixels_per_slice"]))
    )[0]

    if len(valid_z_indices) == 0:
        print(f"[WARN] {patient_id}: valid z slice 없음")
        continue

    valid_z_min = int(valid_z_indices.min())
    valid_z_max = int(valid_z_indices.max())

    patient_patch_count = 0
    patient_candidate_count = 0

    for z in valid_z_indices:
        z = int(z)

        ct_slice = ct_arr[z]
        pure_slice = pure_arr[z]
        organ_slice = organ_arr[z]

        # distance map
        # 폐 mask 안쪽일수록 값이 큼
        dist_map = distance_transform_edt(pure_slice)

        max_dist = float(dist_map.max())
        if max_dist <= 0:
            continue

        dist_ratio_map = dist_map / (max_dist + 1e-6)

        z_level, z_ratio = get_z_level(
            z=z,
            valid_z_min=valid_z_min,
            valid_z_max=valid_z_max,
        )

        for y0 in range(0, h - patch_size + 1, patch_stride):
            for x0 in range(0, w - patch_size + 1, patch_stride):
                patient_candidate_count += 1

                y1 = y0 + patch_size
                x1 = x0 + patch_size

                pure_patch = pure_slice[y0:y1, x0:x1]
                organ_patch = organ_slice[y0:y1, x0:x1]

                pure_pixels = int(pure_patch.sum())
                organ_pixels = int(organ_patch.sum())

                pure_ratio = pure_pixels / float(patch_area)
                organ_ratio = organ_pixels / float(patch_area)

                # 학습용 patch 후보 필터
                if pure_ratio < float(args["min_patch_pure_lung_ratio"]):
                    continue

                if organ_ratio > float(args["max_patch_organ_ratio"]):
                    continue

                ct_patch = ct_slice[y0:y1, x0:x1]

                # central / peripheral 판단
                dist_patch = dist_ratio_map[y0:y1, x0:x1]
                dist_values_inside_pure = dist_patch[pure_patch > 0]

                central_bin, central_ratio_mean = get_central_peripheral(
                    dist_patch_values=dist_values_inside_pure,
                    threshold=float(args["central_distance_ratio_threshold"]),
                )

                position_bin = f"{z_level}_{central_bin}"

                left_right_bin = get_left_right_by_patch_center(
                    x0=x0,
                    patch_size=patch_size,
                    width=w,
                )

                features = calc_patch_features(
                    patch_hu=ct_patch,
                    hist_bins=args["hist_bins"],
                )

                row = {
                    "patient_id": patient_id,
                    "local_z": int(z),
                    "y0": int(y0),
                    "x0": int(x0),
                    "y1": int(y1),
                    "x1": int(x1),

                    "patch_size": int(patch_size),
                    "patch_stride": int(patch_stride),

                    "pure_lung_pixels": int(pure_pixels),
                    "organ_pixels": int(organ_pixels),
                    "pure_lung_patch_ratio": float(pure_ratio),
                    "organ_patch_ratio": float(organ_ratio),

                    "slice_pure_lung_ratio": float(pure_ratio_per_slice[z]),

                    "z_level": z_level,
                    "z_ratio": float(z_ratio),
                    "central_peripheral": central_bin,
                    "central_distance_ratio_mean": float(central_ratio_mean),
                    "position_bin": position_bin,

                    "left_right_metadata": left_right_bin,

                    "ct_1mm_lung_range_nii": str(ct_path),
                    "pure_lung_lung_range_nii": str(pure_path),
                    "organ_exclusion_lung_range_nii": str(organ_path),
                }

                row.update(features)

                all_patch_rows.append(row)
                patient_patch_count += 1

    patient_summary_rows.append({
        "patient_id": patient_id,
        "zdim": int(zdim),
        "height": int(h),
        "width": int(w),
        "valid_z_min": int(valid_z_min),
        "valid_z_max": int(valid_z_max),
        "valid_z_count": int(len(valid_z_indices)),
        "candidate_patch_count_before_filter": int(patient_candidate_count),
        "selected_patch_count": int(patient_patch_count),
    })


patch_df = pd.DataFrame(all_patch_rows)
summary_df = pd.DataFrame(patient_summary_rows)

patch_csv = PATCH_OUT / "metadata_patches.csv"
summary_csv = PATCH_OUT / "patch_patient_summary.csv"

patch_df.to_csv(patch_csv, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

print("\n========== PATCH EXTRACTION FINISHED ==========")
print("patch rows:", len(patch_df))
print("summary rows:", len(summary_df))
print("patch_csv:", patch_csv)
print("summary_csv:", summary_csv)

print("\n========== Patient summary ==========")
display(summary_df)

if len(patch_df) > 0:
    print("\n========== Position bin counts ==========")
    display(patch_df["position_bin"].value_counts())

    print("\n========== Left/right metadata counts ==========")
    display(patch_df["left_right_metadata"].value_counts())

    print("\n========== Patch ratio summary ==========")
    display(
        patch_df[
            [
                "pure_lung_patch_ratio",
                "organ_patch_ratio",
                "slice_pure_lung_ratio",
                "central_distance_ratio_mean",
            ]
        ].describe()
    )
else:
    print("[WARN] 선택된 patch가 없음. ratio 기준을 낮춰야 할 수 있음.")

patient count: 362
patients: ['normal001', 'normal002', 'normal003', 'normal004', 'normal005', 'normal006', 'normal007', 'normal008', 'normal009', 'normal010', 'normal011', 'normal012', 'normal013', 'normal014', 'normal015', 'normal016', 'normal017', 'normal018', 'normal019', 'normal020', 'normal021', 'normal022', 'normal023', 'normal024', 'normal025', 'normal026', 'normal027', 'normal028', 'normal029', 'normal030', 'normal031', 'normal032', 'normal033', 'normal034', 'normal035', 'normal036', 'normal037', 'normal038', 'normal039', 'normal040', 'normal041', 'normal042', 'normal043', 'normal044', 'normal045', 'normal046', 'normal047', 'normal048', 'normal049', 'normal050', 'normal051', 'normal052', 'normal053', 'normal054', 'normal055', 'normal056', 'normal057', 'normal058', 'normal059', 'normal060', 'normal061', 'normal062', 'normal063', 'normal064', 'normal065', 'normal066', 'normal067', 'normal068', 'normal069', 'normal070', 'normal071', 'normal073', 'normal074', 'normal075', 'normal0

Patch extraction by patient:  83%|████████████████████████████████████████▉        | 302/362 [2:08:38<25:33, 25.56s/it]


MemoryError: Unable to allocate 232. MiB for an array with shape (232, 512, 512) and data type float32

In [ ]:
# ============================================================
# Patch visualization check
# ------------------------------------------------------------
# 저장된 metadata_patches.csv에서 patch 몇 개 확인
# ============================================================

from pathlib import Path
import random
import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt
import matplotlib.patches as patches

PRE_ROOT = Path(r"E:\jyp\ct_data_2d_preprocessed\Normal_Cases_final_v1_nonfast_test")
PATCH_OUT = PRE_ROOT / "patch_test_v1"

patch_csv = PATCH_OUT / "metadata_patches.csv"

patch_df = pd.read_csv(patch_csv)

if len(patch_df) == 0:
    raise RuntimeError("metadata_patches.csv가 비어 있음.")

print("patch rows:", len(patch_df))
display(patch_df["position_bin"].value_counts())

# 보고 싶은 position_bin 지정 가능
# None이면 전체에서 랜덤
TARGET_POSITION_BIN = None
# TARGET_POSITION_BIN = "middle_peripheral"

if TARGET_POSITION_BIN is not None:
    view_df = patch_df[patch_df["position_bin"] == TARGET_POSITION_BIN].copy()
else:
    view_df = patch_df.copy()

if len(view_df) == 0:
    raise RuntimeError("해당 조건의 patch가 없음.")

sample_n = min(12, len(view_df))
sample_rows = view_df.sample(sample_n, random_state=42)

# 환자별 CT 배열 캐시
ct_cache = {}
pure_cache = {}

def hu_to_uint8(arr, hu_min=-1000, hu_max=400):
    arr = np.clip(arr, hu_min, hu_max)
    arr = (arr - hu_min) / (hu_max - hu_min + 1e-8)
    arr = (arr * 255.0).astype(np.uint8)
    return arr

cols = 4
rows = int(np.ceil(sample_n / cols))

fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 5))
axes = np.array(axes).reshape(-1)

for ax in axes:
    ax.axis("off")

for i, (_, row) in enumerate(sample_rows.iterrows()):
    ax = axes[i]

    ct_path = row["ct_1mm_lung_range_nii"]
    pure_path = row["pure_lung_lung_range_nii"]

    if ct_path not in ct_cache:
        ct_cache[ct_path] = sitk.GetArrayFromImage(sitk.ReadImage(str(ct_path))).astype(np.float32)

    if pure_path not in pure_cache:
        pure_cache[pure_path] = sitk.GetArrayFromImage(sitk.ReadImage(str(pure_path))) > 0

    ct_arr = ct_cache[ct_path]
    pure_arr = pure_cache[pure_path]

    z = int(row["local_z"])
    y0 = int(row["y0"])
    x0 = int(row["x0"])
    y1 = int(row["y1"])
    x1 = int(row["x1"])

    img = hu_to_uint8(ct_arr[z])
    mask = pure_arr[z]

    ax.imshow(img, cmap="gray")
    ax.imshow(mask, cmap="Reds", alpha=0.25)

    rect = patches.Rectangle(
        (x0, y0),
        x1 - x0,
        y1 - y0,
        linewidth=2,
        edgecolor="yellow",
        facecolor="none",
    )
    ax.add_patch(rect)

    ax.set_title(
        f"{row['patient_id']} z={z}\n"
        f"{row['position_bin']} | {row['left_right_metadata']}\n"
        f"pure={row['pure_lung_patch_ratio']:.2f}, organ={row['organ_patch_ratio']:.2f}",
        fontsize=9,
    )
    ax.axis("off")

plt.tight_layout()
plt.show()